In [ ]:
# 1 - Install qecscreen, pinned by commit SHA (D-028, spec/architecture.md §2).
# QECSCREEN_SHA is the full 40-hex commit to run, never a branch or a tag.
# Cell 2 checks that the installed code really is this commit (D-017).
#
# The repo is private (D-013), so the install reads a read-only GitHub token
# from the platform's secret store, under the name GITHUB_TOKEN:
#   Kaggle: Add-ons -> Secrets, and attach it to this notebook.
#   Colab:  the key icon in the left sidebar, notebook access on.
# Never paste a token into this notebook: its cells and outputs are kept in
# Kaggle's version history and in every copy of it.
import re
import subprocess
import sys

QECSCREEN_SHA = ""
assert re.fullmatch(r"[0-9a-f]{40}", QECSCREEN_SHA), "set QECSCREEN_SHA to a full 40-hex commit"

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except ImportError:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
assert token, "the GITHUB_TOKEN secret is empty or not attached to this notebook"

proc = subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet",
     f"git+https://{token}@github.com/Pushkar0997/qecscreen@{QECSCREEN_SHA}"],
    capture_output=True,
    text=True,
)
print((proc.stdout + proc.stderr).replace(token, "***"))
del token
assert proc.returncode == 0, "pip install failed; its output is above, with the token masked"

In [ ]:
# 2 - Provenance, read back from what is installed, never from the pin above
# (D-017, D-027). record() is every value an artifact carries; none of them is
# a URL or an environment variable, so none can carry the token (D-028).
import importlib

importlib.invalidate_caches()

from qecscreen.provenance import record

PROVENANCE = record()
for key, value in PROVENANCE.items():
    print(f"{key:16} {value}")
assert PROVENANCE["commit_sha"] == QECSCREEN_SHA, "the installed code is not the pinned commit (D-017)"

In [ ]:
# 3 - Configuration: constants only, nothing computed. Protocol values (p, shot
# budgets, decoder settings) are never retyped here; the package reads them
# from qecscreen.protocol.
RUN_NAME = "template-selfcheck"

In [ ]:
# 4 - Output path. Kaggle keeps /kaggle/working as the notebook's output. On
# Colab /content disappears with the runtime: download the artifact.
import datetime
import pathlib

CREATED_AT = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")
KAGGLE_OUT = pathlib.Path("/kaggle/working")
OUT_DIR = KAGGLE_OUT if KAGGLE_OUT.is_dir() else pathlib.Path("/content")
OUT = OUT_DIR / f"{RUN_NAME}-{CREATED_AT.replace(':', '')}.json"
print(OUT)

In [ ]:
# 5 - The one package call. A real run replaces this cell's call and nothing
# else: all logic lives in src/qecscreen (spec/architecture.md §2).
from qecscreen.selfcheck import main as selfcheck

RESULT = {"selfcheck_exit_code": selfcheck()}

In [ ]:
# 6 - Write the artifact and print a summary. Its provenance is record()'s
# dict, whole and unedited.
import json

ARTIFACT = {
    "run_name": RUN_NAME,
    "created_at": CREATED_AT,
    "provenance": PROVENANCE,
    "result": RESULT,
}
OUT.write_text(json.dumps(ARTIFACT, indent=2, sort_keys=True), encoding="utf-8")
print(json.dumps(ARTIFACT, indent=2, sort_keys=True))
print("written:", OUT)